# Process PathoCell Data into CellWhisperer Format

This notebook processes the PathoCell dataset from HDF5 format to AnnData format.
It extracts the first TMA and formats it similar to the lymphoma_cosmx_small dataset.

In [ ]:
import h5py
import numpy as np
import pandas as pd
import anndata
from pathlib import Path
import json
import logging
from PIL import Image
import openslide

logging.basicConfig(level=logging.INFO)

In [ ]:
# Get paths from snakemake
data_dir = Path(snakemake.input.data_dir)
output_adata_path = Path(snakemake.output.adata)
output_image_path = Path(snakemake.output.image)
output_metadata_path = Path(snakemake.output.metadata)

logging.info(f"Reading PathoCell data from {data_dir}")
logging.info(f"Will save processed data to {output_adata_path}")

In [ ]:
# Load the first TMA from PathoCell dataset
pathocell_hdf_dir = data_dir / "pathocell_hdf"
if not pathocell_hdf_dir.exists():
    raise FileNotFoundError(f"pathocell_hdf directory not found in {data_dir}")

hdf_files = list(pathocell_hdf_dir.glob("*.hdf"))
if not hdf_files:
    raise FileNotFoundError(f"No .hdf files found in {pathocell_hdf_dir}")

# Use the first .hdf file (TMA)
hdf_file = hdf_files[0]
logging.info(f"Processing {hdf_file.name}")

with h5py.File(hdf_file, 'r') as f:
    # Explore the structure
    logging.info(f"HDF5 keys: {list(f.keys())}")
    
    # Extract images and masks
    # img: H&E image (3, H, W), gt_inst: instance segmentation (1, H, W), gt_ct: cell type labels (1, H, W)
    if 'img' in f:
        images = f['img'][:]
    elif 'image' in f:
        images = f['image'][:]
    elif 'images' in f:
        image = f['images'][:]
    else:
        raise ValueError("No image data found in HDF5 file")
    
    # Load mask data (cell segmentation)
    # PathoCellBench uses 'gt_inst' for instance segmentation
    if 'gt_inst' in f:
        masks = f['gt_inst'][:]
    elif 'mask' in f:
        masks = f['mask'][:]
    elif 'masks' in f:
        mask = f['masks'][:]
    else:
        raise ValueError("No mask data found in HDF5 file")
    
    # Load labels if available (cell type annotations)
    labels = None
    # PathoCellBench uses 'gt_ct' for cell type annotations
    if 'gt_ct' in f:
        labels = f['gt_ct'][:]
    elif 'labels' in f:
        labels = f['labels'][:]
    elif 'label' in f:
        labels = f['label'][:]
    else:
        raise ValueError("No mask data found in HDF5 file")
    
if labels is not None:
    # Remove the first dimension: (1, H, W) -> (H, W)
    label_map = labels[0]
else:
    label_map = None

# Handle image dimensions
if images is not None:
    # Convert from (3, H, W) to (H, W, 3)
    if len(images.shape) == 3 and images.shape[0] == 3:
        image = np.transpose(images, (1, 2, 0))
    else:
        image = images
else:
    raise ValueError("No image data found in HDF5 file")

# Handle mask dimensions
if masks is not None:
    # Remove the first dimension: (1, H, W) -> (H, W)
    if len(masks.shape) == 3 and masks.shape[0] == 1:
        mask = masks[0]
    else:
        mask = masks
else:
    raise ValueError("No mask data found in HDF5 file")

logging.info(f"Processing image shape {image.shape}, mask shape {mask.shape}")

In [ ]:
# Extract cell centroids from the mask
unique_cell_ids = np.unique(mask)
unique_cell_ids = unique_cell_ids[unique_cell_ids > 0]  # Remove background (0)

logging.info(f"Found {len(unique_cell_ids)} cells")

# Calculate cell centroids and extract labels
cell_data = []
for cell_id in unique_cell_ids:
    cell_mask = (mask == cell_id)
    coords = np.where(cell_mask)
    
    if len(coords[0]) == 0:  # Skip empty masks
        continue
    
    # Calculate centroid
    y_pixel = coords[0].mean()
    x_pixel = coords[1].mean()
    
    # Get cell type label from the label map
    if label_map is not None:
        # Get the label at the cell centroid position
        cell_label_idx = label_map[coords[0][0], coords[1][0]]
        # Convert to string label (assuming labels are categorical)
        cell_type = f"cell_type_{cell_label_idx}"
    else:
        cell_type = "unknown"
    
    cell_data.append({
        'cell_id': str(cell_id),
        'x_pixel': x_pixel,
        'y_pixel': y_pixel,
        'cell_type': cell_type
    })

logging.info(f"Processed {len(cell_data)} valid cells")

In [ ]:
# Create cell metadata DataFrame
cell_df = pd.DataFrame(cell_data)

# Add array coordinates based on patch size (default 224)
patch_size = 224
cell_df['x_array'] = (cell_df['x_pixel'] / patch_size).astype(int)
cell_df['y_array'] = (cell_df['y_pixel'] / patch_size).astype(int)
cell_df['n_cells'] = 1  # Single-cell level

# Set proper index
n_cells = len(cell_df)
cell_df.index = [f"cell_{i}" for i in range(n_cells)]

# Create mock gene expression data (PathoCell is primarily for cell type classification)
# For compatibility with CellWhisperer, we need some expression data
n_genes = 100  # Mock gene set

# Create random expression data (log-normal distribution)
np.random.seed(42)  # For reproducibility
expression_data = np.random.lognormal(mean=1.0, sigma=1.0, size=(n_cells, n_genes))
expression_data = expression_data.astype(np.float32)

# Create gene names
gene_names = [f"GENE_{i:03d}" for i in range(n_genes)]

logging.info(f"Created mock expression matrix: {expression_data.shape}")

In [ ]:
# Create AnnData object
adata = anndata.AnnData(
    X=expression_data,
    obs=cell_df,
    var=pd.DataFrame(index=gene_names)
)

# Add spatial coordinates
adata.obsm['X_spatial'] = cell_df[['x_pixel', 'y_pixel']].values

# Add cell type annotations for classification
adata.obs['cell_type_annotation'] = cell_df['cell_type']

# Add metadata similar to lymphoma_cosmx_small format
library_id = 'pathocell_tma1'
adata.uns['spatial'] = {
    library_id: {
        'images': {},
        'scalefactors': {
            'tissue_hires_scalef': 1.0,
            'tissue_lowres_scalef': 0.1,
            'spot_diameter_fullres': 224,  # Default patch size
        },
        'metadata': {'library_id': library_id},
    }
}

adata.uns['pixel_size'] = 0.5  # Approximate pixel size in micrometers
adata.uns['dataset'] = 'pathocell_tma1'
# Store image_path for CellWhisperer image loading (supports both OpenSlide and PIL)
adata.uns['image_path'] = str(output_image_path)

logging.info(f"Created AnnData with {adata.n_obs} cells and {adata.n_vars} genes")

In [ ]:
# Save the image as TIFF with .svs extension (compatible with OpenSlide)
# SVS format is proprietary, but OpenSlide can read TIFF files with .svs extension
output_image_path.parent.mkdir(parents=True, exist_ok=True)

# Convert to RGB if grayscale
if len(image.shape) == 2:
    image_rgb = np.stack([image, image, image], axis=2)
elif image.shape[2] == 1:
    image_rgb = np.repeat(image, 3, axis=2)
else:
    image_rgb = image

# Ensure data type is uint8 for PIL
if image_rgb.dtype != np.uint8:
    # Normalize to 0-255 range if needed
    if image_rgb.max() <= 1.0:
        image_rgb = (image_rgb * 255).astype(np.uint8)
    else:
        image_rgb = image_rgb.astype(np.uint8)

# Use PIL to save as TIFF with .svs extension (compatible with OpenSlide)
pil_image = Image.fromarray(image_rgb, mode='RGB')
pil_image.save(str(output_image_path), format='TIFF', compression='lzw')
logging.info(f"Saved image as OpenSlide-compatible .svs file to {output_image_path}")

# Validate that the saved file can be read by OpenSlide
slide = openslide.OpenSlide(str(output_image_path))
dimensions = slide.dimensions
slide.close()
logging.info(f"OpenSlide validation successful: image dimensions {dimensions}")

In [ ]:
# Save AnnData object
output_adata_path.parent.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(output_adata_path)
logging.info(f"Saved AnnData to {output_adata_path}")

# Save metadata
metadata = {
    'dataset_name': 'pathocell_tma1',
    'source_file': str(hdf_file),
    'n_cells': int(adata.n_obs),
    'n_genes': int(adata.n_vars),
    'image_shape': list(image.shape),
    'mask_shape': list(mask.shape),
    'unique_cell_types': list(adata.obs['cell_type_annotation'].unique()),
    'pixel_size_um': adata.uns['pixel_size'],
    'processing_info': {
        'mock_expression': True,
        'expression_seed': 42,
        'patch_size': 224
    }
}

output_metadata_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

logging.info(f"Saved metadata to {output_metadata_path}")
logging.info("Processing complete!")